# v2 Agent Smoke Notebook

이 notebook은 v2 Agent API의 기본 동작을 빠르게 확인하기 위한 수동 smoke notebook입니다.

현재 v2 response schema는 MVP 단계의 임시 wrapper이며, 최종 Unreal 연동 규격과 분석 결과 JSON 계약이 확정되면 조정될 수 있습니다.

기본값으로 `V2_AGENT_LLM_ENABLED=false`를 사용하므로 외부 API key 없이 deterministic/rule-based 경로만 확인합니다.

## 1. 프로젝트 루트 설정

In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "app" / "main.py").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

## 2. LLM 비활성 설정

In [ ]:
os.environ["V2_AGENT_LLM_ENABLED"] = "false"
os.environ["V2_AGENT_LLM_REPAIR_ENABLED"] = "true"
os.environ["V2_AGENT_LLM_MAX_REPAIR_ATTEMPTS"] = "1"

{
    "V2_AGENT_LLM_ENABLED": os.environ["V2_AGENT_LLM_ENABLED"],
    "V2_AGENT_LLM_REPAIR_ENABLED": os.environ["V2_AGENT_LLM_REPAIR_ENABLED"],
    "V2_AGENT_LLM_MAX_REPAIR_ATTEMPTS": os.environ["V2_AGENT_LLM_MAX_REPAIR_ATTEMPTS"],
}

## 3. v2 Agent module import 확인

In [ ]:
from app.agents.scenario_generation_v2 import ScenarioGenerationV2Agent
from app.agents.result_analysis_v2 import ResultAnalysisV2Agent
from app.models.scenario_generation_v2 import ScenarioGenerateV2Request
from app.models.analysis_v2 import AnalysisRunV2Request

ScenarioGenerationV2Agent, ResultAnalysisV2Agent, ScenarioGenerateV2Request, AnalysisRunV2Request

## 4. FastAPI app import 및 TestClient 생성

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
client.get("/health").json()

## 5. `/api/v2/scenarios/generate` 정상 요청

In [ ]:
scenario_response = client.post(
    "/api/v2/scenarios/generate",
    json={"prompt": "좁은 보도에서 로봇 전방에 장애물이 있고 보행자가 로봇 경로를 가로지르는 상황"},
)
scenario_payload = scenario_response.json()

assert scenario_response.status_code == 200, scenario_response.text
assert scenario_payload["schema"] == "scenario_generate_response_v2"
assert scenario_payload["generation_mode"] == "deterministic"
assert isinstance(scenario_payload["scenario_template"], dict)

scenario_payload

## 6. 실행 개수 필드 거부 확인

In [ ]:
rejected_fields = {}
for field_name in ["episode_count", "count", "iterations", "run_count"]:
    response = client.post(
        "/api/v2/scenarios/generate",
        json={"prompt": "test", field_name: 1},
    )
    rejected_fields[field_name] = response.status_code
    assert response.status_code == 422, response.text

rejected_fields

## 7. `/api/v2/analysis/run` 기본 요청

In [ ]:
import tempfile

tmp_root = tempfile.TemporaryDirectory()
experiments_root = Path(tmp_root.name) / "experiments"
experiments_root.mkdir()
os.environ["ODIROSIM_EXPERIMENTS_DIR"] = str(experiments_root)

analysis_response = client.post("/api/v2/analysis/run", json={})
analysis_payload = analysis_response.json()

assert analysis_response.status_code == 200, analysis_response.text
assert analysis_payload["schema"] == "analysis_run_response_v2"
assert analysis_payload["analysis_mode"] == "rule_based"

analysis_payload

## 8. 빈 experiments root `insufficient_data` 확인

In [ ]:
assert analysis_payload["analysis_scope"] == {
    "experiments_count": 0,
    "runs_count": 0,
    "episodes_count": 0,
}
assert analysis_payload["summary"]["overall_judgement"] == "insufficient_data"
assert analysis_payload["recommendations"] == []
assert analysis_payload["modified_policy_json"] == []
assert analysis_payload["modified_environment_json"] == []

analysis_payload["summary"]

## 9. 확인 결과 요약

In [ ]:
summary = {
    "scenario_generate_status": scenario_response.status_code,
    "scenario_generation_mode": scenario_payload["generation_mode"],
    "rejected_run_count_fields": rejected_fields,
    "analysis_run_status": analysis_response.status_code,
    "analysis_mode": analysis_payload["analysis_mode"],
    "analysis_judgement": analysis_payload["summary"]["overall_judgement"],
}

summary